In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_deployment='gpt-4o-2024-11-20',
    api_version='2024-08-01-preview'
)

small_llm = AzureChatOpenAI(
    azure_deployment='gpt-4.1-mini',
    api_version='2024-08-01-preview'
)

In [3]:
small_llm.invoke("test")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 8, 'total_tokens': 18, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 23, 'engine_ttft_ms': 39, 'engine_ttlt_ms': 273, 'pre_inference_ms': 156, 'service_tbt_ms': 28, 'service_ttft_ms': 564, 'service_ttlt_ms': 792, 'total_duration_ms': 690, 'user_visible_ttft_ms': 408}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_a7294185dc', 'id': 'chatcmpl-DmaRDdErnDa09JNLQv9NRgCK6NQ9m', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'se

In [18]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """ 숫자 a와 b를 더합니다. """
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """ 숫자 a와 b를 곱합니다. """
    return a * b

In [5]:
add.invoke({'a': 4, 'b': 8})

12

In [6]:
llm_with_tools = small_llm.bind_tools([add, multiply])

In [7]:
query = '3 곱하기 5는?'

In [8]:
small_llm.invoke(query)

AIMessage(content='3 곱하기 5는 15입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 15, 'total_tokens': 27, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 15, 'engine_ttft_ms': 35, 'engine_ttlt_ms': 211, 'pre_inference_ms': 108, 'service_tbt_ms': 23, 'service_ttft_ms': 395, 'service_ttlt_ms': 582, 'total_duration_ms': 559, 'user_visible_ttft_ms': 287}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_a7294185dc', 'id': 'chatcmpl-DmaTItljYc4ocuFEiowGPgaLJsOD0', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filter

In [10]:
result = llm_with_tools.invoke(query)

In [11]:
result.tool_calls

[{'name': 'multiply',
  'args': {'a': 3, 'b': 5},
  'id': 'call_3vRdAhmICKbAi5ptpEHFMyor',
  'type': 'tool_call'}]

In [13]:
from typing import Sequence

from langchain_core.messages import AnyMessage, HumanMessage

human_message = HumanMessage(query)
message_list:  Sequence[AnyMessage] = [human_message]


In [14]:
ai_message = llm_with_tools.invoke(message_list)

In [15]:
ai_message.tool_calls

[{'name': 'multiply',
  'args': {'a': 3, 'b': 5},
  'id': 'call_2JpQ73DKFxxnoHE7AXat8XnD',
  'type': 'tool_call'}]

In [16]:
message_list.append(ai_message)

In [20]:
tool_message = multiply.invoke(ai_message.tool_calls[0])

In [21]:
message_list.append(tool_message)

In [22]:
llm_with_tools.invoke(message_list)

AIMessage(content='3 곱하기 5는 15입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 109, 'total_tokens': 122, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 14, 'engine_ttft_ms': 39, 'engine_ttlt_ms': 220, 'pre_inference_ms': 175, 'service_tbt_ms': 14, 'service_ttft_ms': 426, 'service_ttlt_ms': 583, 'total_duration_ms': 430, 'user_visible_ttft_ms': 251}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b6f445fc1c', 'id': 'chatcmpl-DmaaPllyIhbeAFZsqKMxce9jfre9z', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {}}], 'finish_reason': 'stop', 'logprobs': None, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'